# Initialising packages

In [ ]:
import pandas as pd
import numpy as np
import sklearn as sklearn
import statsmodels.api as stat
import matplotlib.pyplot as plt

In [ ]:
from sklearn.model_selection import LeaveOneOut

# Taking data as input

In [ ]:
gdf=pd.read_csv("gdata.csv")

In [ ]:
gdf

In [ ]:
drugvsgeneexpression=gdf
drugvsgeneexpression=drugvsgeneexpression.dropna(subset=['expressions.GRIA1','expressions.GRIN2A','expressions.GRIN2B','expressions.GRM5','expressions.GRM6','expressions.GRM7','expressions.GRIK1','expressions.DAT','expressions.DRD4','expressions.GABRB3','expressions.GABRG3','expressions.GABRA5', 'pharma.pre.Behavorial Problem','pharma.pre.Inattention','pharma.pre.Hyperactivity','pharma.pre.ADHD Index','pharma.post.Behavorial Problem','pharma.post.Inattention','pharma.post.Hyperactivity','pharma.post.ADHD Index','pharma.Drug'])

In [ ]:
geneexpressionvsconc=gdf
geneexpressionvsconc=geneexpressionvsconc.dropna(subset=['expressions.GRIA1','expressions.GRIN2A','expressions.GRIN2B','expressions.GRM5','expressions.GRM6','expressions.GRM7','expressions.GRIK1','expressions.DAT','expressions.DRD4','expressions.GABRB3','expressions.GABRG3','expressions.GABRA5','concentrations.Serotonin','concentrations.5-HIAA','concentrations.Glutatamate','concentrations.Dopamine','concentrations.GABA'])

In [ ]:
geneexprvstestscore=gdf
geneexprvstestscore=geneexprvstestscore.dropna(subset=['expressions.GRIA1',
'expressions.GRIN2A',
'expressions.GRIN2B',
'expressions.GRM5',
'expressions.GRM6',
'expressions.GRM7',
'expressions.GRIK1',
'expressions.DAT',
'expressions.DRD4',
'expressions.GABRB3'
,'expressions.GABRG3',
'expressions.GABRA5'
,'cprs-r.Behavioral Problem'
,'cprs-r.Inattention'
,'cprs-r.Hyperactivity'
,'cprs-r.ADHD Index',
])

In [ ]:
geneexprvstestscore[['expressions.GRIA1',
'expressions.GRIN2A',
'expressions.GRIN2B',
'expressions.GRM5',
'expressions.GRM6',
'expressions.GRM7',
'expressions.GRIK1',
'expressions.DAT',
'expressions.DRD4',
'expressions.GABRB3'
,'expressions.GABRG3',
'expressions.GABRA5'
,'cprs-r.Behavioral Problem'
,'cprs-r.Inattention'
,'cprs-r.Hyperactivity'
,'cprs-r.ADHD Index',
]] = (geneexprvstestscore[['expressions.GRIA1',
'expressions.GRIN2A',
'expressions.GRIN2B',
'expressions.GRM5',
'expressions.GRM6',
'expressions.GRM7',
'expressions.GRIK1',
'expressions.DAT',
'expressions.DRD4',
'expressions.GABRB3'
,'expressions.GABRG3',
'expressions.GABRA5'
,'cprs-r.Behavioral Problem'
,'cprs-r.Inattention'
,'cprs-r.Hyperactivity'
,'cprs-r.ADHD Index',
]] - geneexprvstestscore[['expressions.GRIA1',
'expressions.GRIN2A',
'expressions.GRIN2B',
'expressions.GRM5',
'expressions.GRM6',
'expressions.GRM7',
'expressions.GRIK1',
'expressions.DAT',
'expressions.DRD4',
'expressions.GABRB3'
,'expressions.GABRG3',
'expressions.GABRA5'
,'cprs-r.Behavioral Problem'
,'cprs-r.Inattention'
,'cprs-r.Hyperactivity'
,'cprs-r.ADHD Index',
]].mean()) / geneexprvstestscore[['expressions.GRIA1',
'expressions.GRIN2A',
'expressions.GRIN2B',
'expressions.GRM5',
'expressions.GRM6',
'expressions.GRM7',
'expressions.GRIK1',
'expressions.DAT',
'expressions.DRD4',
'expressions.GABRB3'
,'expressions.GABRG3',
'expressions.GABRA5'
,'cprs-r.Behavioral Problem'
,'cprs-r.Inattention'
,'cprs-r.Hyperactivity'
,'cprs-r.ADHD Index',
]].std()




# Expressions vs Concentration: OLS models

In [ ]:
I=['expressions.GRIA1',
'expressions.GRIN2A',
'expressions.GRIN2B',
'expressions.GRM5',
'expressions.GRM6',
'expressions.GRM7',
'expressions.GRIK1',
'expressions.DAT',
'expressions.DRD4',
'expressions.GABRB3'
,'expressions.GABRG3',
'expressions.GABRA5']

J=['concentrations.Serotonin',
'concentrations.5-HIAA',
'concentrations.Glutatamate',
'concentrations.Dopamine',
'concentrations.GABA'
]

for i in I:
    for j in J:
        X=geneexpressionvsconc[i]
        Y=geneexpressionvsconc[j]
        model=stat.GLM(Y,X,family=stat.families.Gaussian()).fit()
        print(f"GLM {i} vs {j} summary:{model.summary()}")
        #PLOT
        x_range = np.linspace(X.min(), X.max(), 100)
        y_pred = model.predict(x_range)
        plt.figure(figsize=(10, 6))
        plt.scatter(X, Y, alpha=0.6, label='Observed Data')
        plt.plot(x_range, y_pred, color='red', linewidth=3, label='GLM (Gaussian) Fit')
        plt.title(f'Gaussian GLM: {i} vs {j}')
        plt.xlabel(f'Expression:{i}')
        plt.ylabel(f'Concentration:{j}')
        plt.legend()
        plt.show()


## Leave-one-out method to finding params for LASSO

In [ ]:
print(X)

In [ ]:
X=geneexpressionvsconc[I]
loo = LeaveOneOut()
alphas = np.logspace(-2, 2, 400) # Test 200 values from 0.01 to 10


#J=['concentrations.GABA'] #'concentrations.Serotonin','concentrations.5-HIAA','concentrations.Glutatamate','concentrations.Dopamine','concentrations.GABA'
J=['concentrations.Serotonin',
'concentrations.5-HIAA',
'concentrations.Glutatamate',
'concentrations.Dopamine',
'concentrations.GABA'
]

print(f"Running LOOCV on dataset shape: {X.shape}")

for j in J:
    errors = []
    for a in alphas:
        loo_errors = []
        y=geneexpressionvsconc[j]
        # split() returns indices for training (N-1) and testing (1)
        for train_index, test_index in loo.split(X):
            # Create training and test sets
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train, y_test = y.iloc[train_index], y.iloc[test_index]
            
            try:
                # Fit GLM (Gaussian Family)
                model = stat.OLS(y_train, X_train)
                
                # Regularize (Lasso: L1_wt=1.0)
                result = model.fit_regularized(method='elastic_net', alpha=a, L1_wt=1.0)
                
                # Predict on the single held-out point
                pred = result.predict(X_test)
                
                # Calculate Squared Error
                mse = (y_test.values[0] - pred.values[0])**2
                loo_errors.append(mse)
            except:
                # If model fails to converge (common with tiny data), mark as NaN
                loo_errors.append(np.nan)
        
        # Average the error across all 6 folds for this specific alpha
        errors.append(np.nanmean(loo_errors))

        # Find the best alpha (minimum error)
    best_index = np.argmin(errors)
    best_alpha = alphas[best_index]
    print(f"Optimal Alpha found: {best_alpha:.4f}")

    #PLOT
    plt.figure(figsize=(10, 6))

    # Plot the Error Curve
    plt.plot(alphas, errors, marker='o', color='blue', label='LOOCV MSE')

    # Plot the Vertical Line for Best Alpha
    plt.axvline(best_alpha, color='red', linestyle='--', linewidth=2, label=f'Best Alpha: {best_alpha:.3f}')

    
    plt.xscale('log')
    plt.xlabel('Alpha (Regularization Strength)')
    plt.ylabel('Mean Squared Error (MSE)')
    plt.title(f'Alpha Tuning with Leave-One-Out CV (N={len(X)})')
    plt.legend()
    plt.grid(True, which="both", ls="-", alpha=0.2)

    # Save or Show
    plt.savefig('loocv_plot.png')
    plt.show()

    model1 = stat.GLM(geneexpressionvsconc[j],geneexpressionvsconc[I],family=stat.families.Gaussian()).fit_regularized(method='elastic_net', alpha=best_alpha, L1_wt=1.0)
    print(f"OLS Coefficients for {j}:")
    print(model1.params)


In [ ]:
X=geneexpressionvsconc[I]
loo = LeaveOneOut()
alphas = np.logspace(-2, 1, 400) # Test 200 values from 0.01 to 10


#J=['concentrations.GABA'] #'concentrations.Serotonin','concentrations.5-HIAA','concentrations.Glutatamate','concentrations.Dopamine','concentrations.GABA'
J=['concentrations.Serotonin',
'concentrations.5-HIAA',
'concentrations.Glutatamate',
'concentrations.Dopamine',
'concentrations.GABA'
]

print(f"Running LOOCV on dataset shape: {X.shape}")

for j in J:
    errors = []
    for a in alphas:
        loo_errors = []
        y=geneexpressionvsconc[j]
        # split() returns indices for training (N-1) and testing (1)
        for train_index, test_index in loo.split(X):
            # Create training and test sets
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train, y_test = y.iloc[train_index], y.iloc[test_index]
            
            try:
                # Fit GLM (Gaussian Family)
                model = stat.OLS(y_train, X_train)
                
                # Regularize (Lasso: L1_wt=1.0)
                result = model.fit_regularized(method='elastic_net', alpha=a, L1_wt=1.0)
                
                # Predict on the single held-out point
                pred = result.predict(X_test)
                
                # Calculate Squared Error
                mse = (y_test.values[0] - pred.values[0])**2
                loo_errors.append(mse)
            except:
                # If model fails to converge (common with tiny data), mark as NaN
                loo_errors.append(np.nan)
        
        # Average the error across all 6 folds for this specific alpha
        errors.append(np.nanmean(loo_errors))

        # Find the best alpha (minimum error)
    best_index = np.argmin(errors)
    best_alpha = alphas[best_index]
    print(f"Optimal Alpha found: {best_alpha:.4f}")

    #PLOT
    plt.figure(figsize=(10, 6))

    # Plot the Error Curve
    plt.plot(alphas, errors, marker='o', color='blue', label='LOOCV MSE')

    # Plot the Vertical Line for Best Alpha
    plt.axvline(best_alpha, color='red', linestyle='--', linewidth=2, label=f'Best Alpha: {best_alpha:.3f}')

    
    plt.xscale('log')
    plt.xlabel('Alpha (Regularization Strength)')
    plt.ylabel('Mean Squared Error (MSE)')
    plt.title(f'Alpha Tuning with Leave-One-Out CV (N={len(X)})')
    plt.legend()
    plt.grid(True, which="both", ls="-", alpha=0.2)

    # Save or Show
    plt.savefig('loocv_plot.png')
    plt.show()

    model1 = stat.GLM(geneexpressionvsconc[j],geneexpressionvsconc[I],family=stat.families.Gaussian()).fit_regularized(method='elastic_net', alpha=best_alpha, L1_wt=1.0)
    print(f"OLS Coefficients for {j}:")
    print(model1.params)


# Expressions vs Test score(without drug effect)

In [ ]:
I=['expressions.GRIA1',
'expressions.GRIN2A',
'expressions.GRIN2B',
'expressions.GRM5',
'expressions.GRM6',
'expressions.GRM7',
'expressions.GRIK1',
'expressions.DAT',
'expressions.DRD4',
'expressions.GABRB3'
,'expressions.GABRG3',
'expressions.GABRA5']

J=['cprs-r.Behavioral Problem'
,'cprs-r.Inattention'
,'cprs-r.Hyperactivity'
,'cprs-r.ADHD Index']

for i in I:
    for j in J:
        X=geneexprvstestscore[i]
        Y=geneexprvstestscore[j]
        model=stat.GLM(Y,X,family=stat.families.Gaussian()).fit()
        print(f"GLM {i} vs {j} summary:{model.summary()}")
        #PLOT
        x_range = np.linspace(X.min(), X.max(), 100)
        y_pred = model.predict(x_range)
        plt.figure(figsize=(10, 6))
        plt.scatter(X, Y, alpha=0.6, label='Observed Data')
        plt.plot(x_range, y_pred, color='red', linewidth=3, label='GLM (Gaussian) Fit')
        plt.title(f'Gaussian GLM: {i} vs {j}')
        plt.xlabel(f'Expression:{i}')
        plt.ylabel(f'Test:{j}')
        plt.legend()
        plt.show()

for j in J:
    model=stat.GLM(geneexprvstestscore[j],geneexprvstestscore[I],family=stat.families.Gaussian()).fit()
    print(f"GLM all expressions vs {j} summary: {model.summary()}")


# Expression vs test score(with drug effect)

In [ ]:
I=['expressions.GRIA1',
'expressions.GRIN2A',
'expressions.GRIN2B',
'expressions.GRM5',
'expressions.GRM6',
'expressions.GRM7',
'expressions.GRIK1',
'expressions.DAT',
'expressions.DRD4',
'expressions.GABRB3'
,'expressions.GABRG3',
'expressions.GABRA5']

J1=['pharma.pre.Behavorial Problem',
'pharma.pre.Inattention',
'pharma.pre.Hyperactivity',
'pharma.pre.ADHD Index']

J2=['pharma.post.Behavorial Problem',
'pharma.post.Inattention',
'pharma.post.Hyperactivity',
'pharma.post.ADHD Index'
]

drugdat=drugvsgeneexpression[drugvsgeneexpression['pharma.Drug']=='Methylphenidate']

for i in I:
    for j1,j2 in zip(J1,J2):
        X=drugdat[i]
        Y=drugdat[j2]-drugdat[j1]
        X=(X-X.mean())/X.std()
        X=stat.add_constant(X)
        model=stat.OLS(Y,X).fit()
        print(f"OLS {i} vs {j2}-{j1} summary:{model.summary()}")
        #PLOT
        x_range = np.linspace(X.min(), X.max(), 100)
        y_pred = model.predict(x_range)
        plt.figure(figsize=(10, 6))
        plt.scatter(X[i], Y, alpha=0.6, label='Observed Data')
        plt.plot(x_range, y_pred, color='red', linewidth=3, label='OLS Fit')
        plt.title(f'OLS: {i} vs {j2}-{j1}')
        plt.xlabel(f'Expression:{i}')
        plt.ylabel(f'Test score difference:{j2}-{j1}')
        plt.legend()
        plt.show()

# for j1,j2 in zip(J1,J2):
#     model=stat.(geneexprvstestscore[j],geneexprvstestscore[I],family=stat.families.Gaussian()).fit()
#     print(f"GLM all expressions vs {j} summary: {model.summary()}")


## Leave-one-method to find best parameter for LASSO 

In [ ]:
X=drugdat[I]
X=(X-X.mean())/X.std()
X=stat.add_constant(X)

loo = LeaveOneOut()
alphas = np.logspace(-2, 1, 200) # Test 200 values from 0.01 to 10





print(f"Running LOOCV on dataset shape: {X.shape}")

for j1,j2 in zip(J1,J2):
    y=drugdat[j2]-drugdat[j1]
    errors = []
    for a in alphas:
        loo_errors = []
        
        # split() returns indices for training (N-1) and testing (1)
        for train_index, test_index in loo.split(X):
            # Create training and test sets
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train, y_test = y.iloc[train_index], y.iloc[test_index]
            
            try:
                # Fit GLM (Gaussian Family)
                model = stat.OLS(y_train, X_train)
                
                # Regularize (Lasso: L1_wt=1.0)
                result = model.fit_regularized(method='elastic_net', alpha=a, L1_wt=1.0)
                
                # Predict on the single held-out point
                pred = result.predict(X_test)
                
                # Calculate Squared Error
                mse = (y_test.values[0] - pred.values[0])**2
                loo_errors.append(mse)
            except:
                # If model fails to converge (common with tiny data), mark as NaN
                loo_errors.append(np.nan)
        
        # Average the error across all 6 folds for this specific alpha
        errors.append(np.nanmean(loo_errors))

        # Find the best alpha (minimum error)
    best_index = np.argmin(errors)
    best_alpha = alphas[best_index]
    print(f"Optimal Alpha found: {best_alpha:.4f}")

    #PLOT
    plt.figure(figsize=(10, 6))

    # Plot the Error Curve
    plt.plot(alphas, errors, marker='o', color='blue', label='LOOCV MSE')

    # Plot the Vertical Line for Best Alpha
    plt.axvline(best_alpha, color='red', linestyle='--', linewidth=2, label=f'Best Alpha: {best_alpha:.3f}')

    
    plt.xscale('log')
    plt.xlabel('Alpha (Regularization Strength)')
    plt.ylabel('Mean Squared Error (MSE)')
    plt.title(f'Alpha Tuning with Leave-One-Out CV (N={len(X)})')
    plt.legend()
    plt.grid(True, which="both", ls="-", alpha=0.2)

    # Save or Show
    plt.savefig('loocv_plot.png')
    plt.show()

    model1 = stat.OLS(y,X).fit_regularized(method='elastic_net', alpha=best_alpha, L1_wt=1.0)
    print(f"OLS Coefficients for {j1}:")
    print(model1.params)
